In [ ]:
import pandas as pd
from Bio.SeqUtils.ProtParam import ProteinAnalysis


input_files = {
    "CU": "CU-Copper_Other_Interactions_3A-Electronegative-processed.csv",
    "CU1": "CU1-Copper_Other_Interactions_3A-Electronegative-processed.csv"
}


three_to_one_letter = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLU': 'E', 'GLN': 'Q', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
}


isoelectric_points = {
    'ALA': 6.00, 'ARG': 10.76, 'ASN': 5.41, 'ASP': 2.77, 'CYS': 5.07,
    'GLN': 5.65, 'GLU': 3.22, 'GLY': 5.97, 'HIS': 7.59, 'ILE': 6.02,
    'LEU': 5.98, 'LYS': 9.74, 'MET': 5.74, 'PHE': 5.48, 'PRO': 6.30,
    'SER': 5.68, 'THR': 5.60, 'TRP': 5.89, 'TYR': 5.66, 'VAL': 5.96,
    'HOH': 7.0, 'OH': 7.0, 'O': 7.0,
    'OTHER': 0.0
}


amino_acid_classification = {
    'A': 'Non-polar', 'G': 'Non-polar', 'I': 'Non-polar', 'L': 'Non-polar',
    'M': 'Non-polar', 'W': 'Non-polar', 'F': 'Non-polar', 'P': 'Non-polar',
    'V': 'Non-polar',
    'C': 'Polar neutral', 'S': 'Polar neutral', 'T': 'Polar neutral',
    'Y': 'Polar neutral', 'N': 'Polar neutral', 'Q': 'Polar neutral',
    'H': 'Polar basic', 'K': 'Polar basic', 'R': 'Polar basic',
    'D': 'Polar acidic', 'E': 'Polar acidic'
}


def get_aa_properties(res_name):
    if not isinstance(res_name, str):
        res_name = "OTHER"

    res_name = res_name.upper()

    if res_name == "OTHER":
        return {
            'Molecular_Weight': 0.0,
            'Isoelectric_Point': 0.0,
            'Hydrophobicity': 0.0,
            'Classification': 'OTHER'
        }

    if res_name in {'HOH', 'OH', 'O'}:
        return {
            'Molecular_Weight': 18.01528,
            'Isoelectric_Point': isoelectric_points[res_name],
            'Hydrophobicity': 0.0,
            'Classification': 'Polar O'
        }

    one_letter = three_to_one_letter.get(res_name)

    if one_letter is None:
        return {
            'Molecular_Weight': 0.0,
            'Isoelectric_Point': 0.0,
            'Hydrophobicity': 0.0,
            'Classification': 'OTHER'
        }

    analysis = ProteinAnalysis(one_letter)

    return {
        'Molecular_Weight': analysis.molecular_weight(),
        'Isoelectric_Point': isoelectric_points[res_name],
        'Hydrophobicity': analysis.gravy(),
        'Classification': amino_acid_classification.get(one_letter, 'OTHER')
    }


def process_features(input_file, output_file):

    df = pd.read_csv(input_file)

    df = df.fillna("OTHER")
    df = df.replace("Other", "OTHER")


    # Extract residue physicochemical properties: molecular weight, isoelectric point, hydrophobicity, and amino-acid classification.
    res_cols = [
        'res_1', 'res_2', 'res_3',
        'res_4', 'res_5', 'res_6', 'res_7'
    ]

    feature_rows = []

    for _, row in df.iterrows():
        feature_row = {}

        for col in res_cols:
            props = get_aa_properties(row[col])

            for prop_name, value in props.items():
                feature_row[f"{col}_{prop_name}"] = value

        feature_rows.append(feature_row)

    features_df = pd.DataFrame(feature_rows)

    df = pd.concat([df, features_df], axis=1)


    # Extract the number of aromatic residues (PHE, TYR, and TRP).
    aromatic_residues = {"PHE", "TYR", "TRP"}

    def count_aromatic(row):
        count = 0

        for col in res_cols:
            val = row[col]

            if not isinstance(val, str):
                continue

            if val.upper() == "OTHER":
                continue

            res_name = val.split("_")[0].upper()

            if res_name in aromatic_residues:
                count += 1

        return count

    df["Aromatic_Count"] = df.apply(count_aromatic, axis=1)


    # Extract water count, individual amino-acid counts, and residue count.
    residue_columns = [
        'res_1', 'res_2', 'res_3',
        'res_4', 'res_5', 'res_6', 'res_7'
    ]

    water_set = {'HOH', 'OH', 'O'}

    water_count = (
        df[residue_columns]
        .applymap(lambda x: isinstance(x, str) and x in water_set)
        .sum(axis=1)
    )

    amino_acid_residues = df[residue_columns].applymap(
        lambda x: x
        if isinstance(x, str) and x not in {'OTHER', 'HOH', 'OH', 'O'}
        else None
    )

    combined_amino_acids = (
        amino_acid_residues
        .apply(lambda x: '_'.join(x.dropna()), axis=1)
        .str.split('_')
        .explode()
        .dropna()
        .reset_index()
    )

    amino_acid_counts = (
        combined_amino_acids
        .groupby('index')[0]
        .value_counts()
        .unstack(fill_value=0)
    )

    amino_acid_counts.columns = [
        f'{col}_Count'
        for col in amino_acid_counts.columns
    ]

    df['Unique_Residue_Count'] = (
        df[residue_columns]
        .applymap(
            lambda x: isinstance(x, str) and x.upper() != "OTHER"
        )
        .sum(axis=1)
    )

    df = pd.concat([df, amino_acid_counts], axis=1)
    df['Water_Count'] = water_count


    # Extract generalized coordinating-atom counts from the atom columns.
    atom_columns = [
        'Atom_1', 'Atom_2', 'Atom_3', 'Atom_4',
        'Atom_5', 'Atom_6', 'Atom_7', 'Atom_8'
    ]

    generalized_atoms = (
        df[atom_columns]
        .replace("OTHER", pd.NA)
        .apply(lambda col: col.str.replace(r'\d+', '', regex=True))
    )

    atoms_long = (
        generalized_atoms
        .stack()
        .dropna()
    )

    atom_counts = (
        atoms_long
        .groupby(level=0)
        .value_counts()
        .unstack(fill_value=0)
    )

    atom_counts.columns = [
        f'Atom_{col}'
        for col in atom_counts.columns
    ]

    df = df.join(atom_counts)


    # Extract counts of residue chemical classes: non-polar, acidic, basic, neutral, and polar oxygen.
    class_nature_columns = [
        'res_1_Classification', 'res_2_Classification',
        'res_3_Classification', 'res_4_Classification',
        'res_5_Classification', 'res_6_Classification',
    ]

    valid_classes = {
        'Non-polar',
        'Polar acidic',
        'Polar basic',
        'Polar neutral',
        'Polar O'
    }

    residue_classes = df[class_nature_columns].where(
        df[class_nature_columns].isin(valid_classes)
    )

    class_nature_counts = (
        residue_classes
        .stack()
        .groupby(level=0)
        .value_counts()
        .unstack(fill_value=0)
    )

    expected_classes = [
        'Non-polar',
        'Polar acidic',
        'Polar basic',
        'Polar neutral',
        'Polar O'
    ]

    class_nature_counts = class_nature_counts.reindex(
        columns=expected_classes,
        fill_value=0
    )

    class_nature_counts.columns = [
        f'Class_Nature_{c}'
        for c in class_nature_counts.columns
    ]

    df = df.join(class_nature_counts)


    # Extract average molecular weight, isoelectric point, and hydrophobicity across residues.
    df['Sum_Molecular_Weight'] = df[
        [f'res_{i}_Molecular_Weight' for i in range(1, 8)]
    ].sum(axis=1)

    df['Sum_Isoelectric_Point'] = df[
        [f'res_{i}_Isoelectric_Point' for i in range(1, 8)]
    ].sum(axis=1)

    df['Sum_Hydrophobicity'] = df[
        [f'res_{i}_Hydrophobicity' for i in range(1, 8)]
    ].sum(axis=1)

    df['Average_Molecular_Weight'] = (
        df['Sum_Molecular_Weight'] /
        df['Unique_Residue_Count']
    )

    df['Average_Isoelectric_Point'] = (
        df['Sum_Isoelectric_Point'] /
        df['Unique_Residue_Count']
    )

    df['Average_Hydrophobicity'] = (
        df['Sum_Hydrophobicity'] /
        df['Unique_Residue_Count']
    )

    df.drop(
        columns=[
            'Sum_Molecular_Weight',
            'Sum_Isoelectric_Point',
            'Sum_Hydrophobicity'
        ],
        inplace=True
    )


    # Extract consolidated nitrogen, sulfur, and oxygen coordinating-atom counts.
    df['Comb-Atom_N'] = df[
        ['Atom_N', 'Atom_ND', 'Atom_NE', 'Atom_NH', 'Atom_NZ']
    ].sum(axis=1)

    df['Comb-Atom_S'] = df[
        ['Atom_SG', 'Atom_SD']
    ].sum(axis=1)

    df['Comb-Atom_O'] = df[
        ['Atom_O', 'Atom_OD', 'Atom_OE', 'Atom_OG', 'Atom_OH']
    ].sum(axis=1)


    df.to_csv(output_file, index=False)

    print(f"Processed: {input_file}")
    print(f"Saved: {output_file}")


for metal_type, input_file in input_files.items():

    output_file = f"{metal_type}-Final-features-updated.csv"

    process_features(
        input_file,
        output_file
    )

In [ ]:
# Combine the Cu1 and Cu2 feature datasets.
cu1_df = pd.read_csv("CU-Final-features-updated.csv")
cu2_df = pd.read_csv("CU1-Final-features-updated.csv")

df = pd.concat(
    [cu1_df, cu2_df],
    ignore_index=True
)


# Extract the residue combination from individual amino-acid and water counts.
residue_count_cols = [
    'ARG_Count', 'ASN_Count', 'ASP_Count', 'CYS_Count',
    'GLN_Count', 'GLU_Count', 'GLY_Count', 'HIS_Count',
    'MET_Count', 'SER_Count', 'THR_Count', 'TYR_Count',
    'VAL_Count', 'Water_Count', 'PHE_Count', 'ILE_Count',
    'LEU_Count', 'ALA_Count', 'LYS_Count', 'TRP_Count'
]

def get_residue_combination(row):
    return ' '.join([
        f"{int(row[col])} {col.split('_')[0]}"
        for col in residue_count_cols
        if col in row and row[col] > 0
    ])

df["Residue_Combination"] = df.apply(
    get_residue_combination,
    axis=1
)


# Remove rows containing only water and no amino-acid residues.
non_water_cols = [
    col for col in residue_count_cols
    if col != "Water_Count"
]

df_filtered = df[
    ~(
        (df["Water_Count"] > 0) &
        (df[non_water_cols].sum(axis=1) == 0)
    )
]


# Save the final combined dataset.
df_filtered.to_csv("input.csv", index=False)